In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from itertools import permutations
from ipywidgets import Dropdown, HTML, VBox, HBox, Layout
from IPython.display import display

# ============================================================
# CASCADE IMPLEMENTATION — SECOND-ORDER SECTIONS
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12.5,'axes.labelsize':10.5,'xtick.labelsize':9.5,'ytick.labelsize':9.5,'legend.fontsize':8.8})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.cas-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.cas-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.cas-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:9px 13px;
    border-radius:0 0 8px 8px;
    font-size:14px;
    line-height:1.48;
    margin-bottom:7px;
}

.cas-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:8px 11px;
    margin-bottom:6px;
    font-size:13.5px;
    line-height:1.45;
}

.cas-result{
    background:#fff8e6;
    border:1px solid #d8b451;
}

.cas-title{
    color:#0d47a1;
    font-weight:bold;
    font-size:14.5px;
    margin-bottom:5px;
}

.cas-equation{
    text-align:center;
    font-family:serif;
    font-size:16px;
    margin:6px 0;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="cas-root">

<div class="cas-header">
Cascade Implementation — From H(z) to Second-Order Sections
</div>

<div class="cas-doc">

A high-order IIR transfer function can be factored into a product of
first- and second-order transfer functions. For a sixth-order system,

<div class="cas-equation">
<b>
H(z) = H₁(z) H₂(z) H₃(z).
</b>
</div>

Each second-order subsystem is implemented independently and the output of
one section becomes the input of the next.

<div class="cas-equation">
x[n] → H₁(z) → H₂(z) → H₃(z) → y[n]
</div>

The order of the sections may be changed without changing the theoretical
overall transfer function. However, the internal signals of the cascade
change because each section receives a different intermediate input.

This notebook demonstrates three ideas simultaneously:

<b>factorization → cascade implementation → input/output equivalence.</b>

</div>

</div>
"""))

# ============================================================
# FIXED SIXTH-ORDER IIR FILTER
# ============================================================

ORDER = 6
CUTOFF = 0.35

b,a = signal.butter(ORDER,CUTOFF,btype='low',output='ba')

# Automatically convert the complete transfer function
# into second-order sections.
sos = signal.tf2sos(b,a)

NUM_SECTIONS = sos.shape[0]

# ============================================================
# FREQUENCY RESPONSE OF EACH ORIGINAL SECTION
# ============================================================

omega = np.linspace(0.0,np.pi,1024)

section_H = []

for k in range(NUM_SECTIONS):

    _,Hk = signal.freqz(sos[k,:3],sos[k,3:],worN=omega)

    section_H.append(Hk)

section_H = np.array(section_H)

# ============================================================
# TEST SIGNAL
# ============================================================

N = 80

n = np.arange(N)

x = np.zeros(N)

x[0] = 1.0

x += 0.30*np.sin(0.22*np.pi*n)

x += 0.15*np.sin(0.72*np.pi*n)

# Direct-form reference output.
y_direct = signal.lfilter(b,a,x)

# ============================================================
# ALL POSSIBLE SECTION ORDERS
# ============================================================

all_orders = list(permutations(range(NUM_SECTIONS)))

order_options = []

for order in all_orders:

    label = ' → '.join([f'H{k+1}' for k in order])

    order_options.append((label,order))

order_dropdown = Dropdown(options=order_options,value=all_orders[0],description='Section order:',style={'description_width':'95px'},layout=Layout(width='370px'))

controls = HBox([order_dropdown],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b9cce5',padding='6px 9px',margin='0 0 3px 0'))

# ============================================================
# PRECOMPUTE OUTPUT AXIS RANGE FOR ALL POSSIBLE ORDERS
# ============================================================

all_stage_values = []

all_differences = []

for order in all_orders:

    u = x.copy()

    for idx in order:

        section = sos[idx]

        u = signal.lfilter(section[:3],section[3:],u)

        all_stage_values.extend(u)

    all_differences.extend(u-y_direct)

stage_min = min(np.min(all_stage_values),np.min(y_direct))

stage_max = max(np.max(all_stage_values),np.max(y_direct))

stage_margin = 0.08*(stage_max-stage_min)

OUTPUT_YMIN = stage_min-stage_margin

OUTPUT_YMAX = stage_max+stage_margin

difference_abs_max = max(np.max(np.abs(all_differences)),1e-15)

DIFF_LIMIT = 1.20*difference_abs_max

# ============================================================
# INITIAL ORDER
# ============================================================

current_order = order_dropdown.value

# ============================================================
# INITIAL CASCADE CALCULATION
# ============================================================

def calculate_cascade(order):

    stage_outputs = []

    u = x.copy()

    for idx in order:

        section = sos[idx]

        u = signal.lfilter(section[:3],section[3:],u)

        stage_outputs.append(u.copy())

    return stage_outputs

initial_outputs = calculate_cascade(current_order)

y_cascade = initial_outputs[-1]

difference = y_direct-y_cascade

# ============================================================
# NUMERICAL INFORMATION
# ============================================================

result_html = HTML(layout=Layout(width=CONTENT_WIDTH))

# ============================================================
# FIGURE 1 — CASCADE STRUCTURE
# CREATED ONCE
# ============================================================

fig1,ax_structure = plt.subplots(figsize=(9.0,2.25))

fig1.canvas.toolbar_visible = False
fig1.canvas.header_visible = False
fig1.canvas.footer_visible = False

ax_structure.set_xlim(0,10)

ax_structure.set_ylim(0,3)

ax_structure.axis('off')

ax_structure.set_title('Cascade Structure of the Sixth-Order Filter')

box_y = 1.5

box_width = 1.55

box_height = 0.80

box_centers = [3.0,5.0,7.0]

# Input arrow
ax_structure.annotate('',xy=(2.18,box_y),xytext=(0.75,box_y),arrowprops={'arrowstyle':'->','linewidth':1.5})

ax_structure.text(0.55,box_y+0.22,r'$x[n]$',fontsize=11,fontweight='bold')

# Section boxes
section_boxes = []

section_texts = []

for center in box_centers:

    rect = plt.Rectangle((center-box_width/2,box_y-box_height/2),box_width,box_height,fill=False,linewidth=1.4)

    ax_structure.add_patch(rect)

    section_boxes.append(rect)

    text = ax_structure.text(center,box_y,'',ha='center',va='center',fontsize=11,fontweight='bold')

    section_texts.append(text)

# Connecting arrows
ax_structure.annotate('',xy=(4.18,box_y),xytext=(3.82,box_y),arrowprops={'arrowstyle':'->','linewidth':1.5})

ax_structure.annotate('',xy=(6.18,box_y),xytext=(5.82,box_y),arrowprops={'arrowstyle':'->','linewidth':1.5})

# Output arrow
ax_structure.annotate('',xy=(9.25,box_y),xytext=(7.82,box_y),arrowprops={'arrowstyle':'->','linewidth':1.5})

ax_structure.text(9.45,box_y+0.22,r'$y[n]$',fontsize=11,fontweight='bold',ha='center')

plt.subplots_adjust(left=0.03,right=0.98,top=0.82,bottom=0.08)

# ============================================================
# FIGURE 2 — FREQUENCY RESPONSES
# CREATED ONCE
# ============================================================

fig2,(ax_sections,ax_cumulative) = plt.subplots(1,2,figsize=(9.0,3.6))

fig2.canvas.toolbar_visible = False
fig2.canvas.header_visible = False
fig2.canvas.footer_visible = False

# ------------------------------------------------------------
# INDIVIDUAL SECTIONS
# ------------------------------------------------------------

individual_lines = []

for k in range(NUM_SECTIONS):

    mag = 20*np.log10(np.maximum(np.abs(section_H[k]),1e-12))

    line, = ax_sections.plot(omega/np.pi,mag,linewidth=1.3,label=f'Section {k+1}')

    individual_lines.append(line)

ax_sections.set_xlim(0,1)

ax_sections.set_ylim(-160,40)

ax_sections.set_title('Individual Section Responses')

ax_sections.set_xlabel(r'Normalized frequency $\omega/\pi$')

ax_sections.set_ylabel('Magnitude (dB)')

ax_sections.grid(True,linestyle=':',alpha=0.30)

ax_sections.legend(loc='lower left')

# ------------------------------------------------------------
# CUMULATIVE RESPONSES
# ------------------------------------------------------------

cumulative_lines = []

initial_cumulative = np.ones_like(section_H[0],dtype=complex)

for stage,idx in enumerate(current_order):

    initial_cumulative *= section_H[idx]

    mag = 20*np.log10(np.maximum(np.abs(initial_cumulative),1e-12))

    line, = ax_cumulative.plot(omega/np.pi,mag,linewidth=1.3,label=f'After stage {stage+1}')

    cumulative_lines.append(line)

ax_cumulative.set_xlim(0,1)

ax_cumulative.set_ylim(-160,40)

ax_cumulative.set_title('Cumulative Cascade Response')

ax_cumulative.set_xlabel(r'Normalized frequency $\omega/\pi$')

ax_cumulative.set_ylabel('Magnitude (dB)')

ax_cumulative.grid(True,linestyle=':',alpha=0.30)

ax_cumulative.legend(loc='lower left')

plt.subplots_adjust(left=0.08,right=0.98,top=0.88,bottom=0.16,wspace=0.28)

# ============================================================
# FIGURE 3 — TIME-DOMAIN OUTPUTS
# CREATED ONCE
# ============================================================

fig3,(ax_stages,ax_compare) = plt.subplots(1,2,figsize=(9.0,3.6))

fig3.canvas.toolbar_visible = False
fig3.canvas.header_visible = False
fig3.canvas.footer_visible = False

# ------------------------------------------------------------
# STAGE OUTPUTS
# ------------------------------------------------------------

stage_lines = []

for k in range(NUM_SECTIONS):

    line, = ax_stages.plot(n,initial_outputs[k],linewidth=1.2,label=f'Stage {k+1}')

    stage_lines.append(line)

ax_stages.set_xlim(0,N-1)

ax_stages.set_ylim(OUTPUT_YMIN,OUTPUT_YMAX)

ax_stages.set_title('Intermediate Cascade Signals')

ax_stages.set_xlabel('Sample index n')

ax_stages.set_ylabel('Amplitude')

ax_stages.grid(True,linestyle=':',alpha=0.30)

ax_stages.legend(loc='upper right')

# ------------------------------------------------------------
# FINAL OUTPUT COMPARISON
# ------------------------------------------------------------

direct_line, = ax_compare.plot(n,y_direct,linewidth=1.5,label='Direct implementation')

cascade_line, = ax_compare.plot(n,y_cascade,'--',linewidth=1.3,label='Cascade implementation')

ax_compare.set_xlim(0,N-1)

ax_compare.set_ylim(OUTPUT_YMIN,OUTPUT_YMAX)

ax_compare.set_title('Final Output Comparison')

ax_compare.set_xlabel('Sample index n')

ax_compare.set_ylabel('y[n]')

ax_compare.grid(True,linestyle=':',alpha=0.30)

ax_compare.legend(loc='upper right')

plt.subplots_adjust(left=0.08,right=0.98,top=0.88,bottom=0.16,wspace=0.28)

# ============================================================
# FIGURE 4 — NUMERICAL DIFFERENCE
# CREATED ONCE
# ============================================================

fig4,ax_difference = plt.subplots(figsize=(9.0,2.6))

fig4.canvas.toolbar_visible = False
fig4.canvas.header_visible = False
fig4.canvas.footer_visible = False

difference_line, = ax_difference.plot(n,difference,linewidth=1.2)

ax_difference.axhline(0,linewidth=0.8)

ax_difference.set_xlim(0,N-1)

ax_difference.set_ylim(-DIFF_LIMIT,DIFF_LIMIT)

ax_difference.set_title('Numerical Difference: Direct − Cascade')

ax_difference.set_xlabel('Sample index n')

ax_difference.set_ylabel('Difference')

ax_difference.grid(True,linestyle=':',alpha=0.30)

plt.subplots_adjust(left=0.10,right=0.98,top=0.84,bottom=0.22)

# ============================================================
# UPDATE FUNCTION
# ============================================================

def update(change=None):

    order = order_dropdown.value

    # --------------------------------------------------------
    # UPDATE STRUCTURE LABELS
    # --------------------------------------------------------

    for stage,idx in enumerate(order):

        section_texts[stage].set_text(rf'$H_{{{idx+1}}}(z)$')

    # --------------------------------------------------------
    # CALCULATE CASCADE OUTPUT
    # --------------------------------------------------------

    outputs = calculate_cascade(order)

    cascade_output = outputs[-1]

    diff = y_direct-cascade_output

    # --------------------------------------------------------
    # UPDATE CUMULATIVE FREQUENCY RESPONSES
    # --------------------------------------------------------

    cumulative = np.ones_like(section_H[0],dtype=complex)

    for stage,idx in enumerate(order):

        cumulative *= section_H[idx]

        mag = 20*np.log10(np.maximum(np.abs(cumulative),1e-12))

        cumulative_lines[stage].set_ydata(mag)

        cumulative_lines[stage].set_label(f'After {stage+1}: H{idx+1}')

    ax_cumulative.legend(loc='lower left')

    # --------------------------------------------------------
    # UPDATE INTERMEDIATE OUTPUTS
    # --------------------------------------------------------

    for stage in range(NUM_SECTIONS):

        stage_lines[stage].set_ydata(outputs[stage])

        stage_lines[stage].set_label(f'After H{order[stage]+1}')

    ax_stages.legend(loc='upper right')

    # --------------------------------------------------------
    # UPDATE FINAL CASCADE OUTPUT
    # --------------------------------------------------------

    cascade_line.set_ydata(cascade_output)

    difference_line.set_ydata(diff)

    # --------------------------------------------------------
    # NUMERICAL VERIFICATION
    # --------------------------------------------------------

    maximum_difference = np.max(np.abs(diff))

    order_text = ' → '.join([f'H{k+1}' for k in order])

    section_rows = ""

    for stage,idx in enumerate(order):

        sec = sos[idx]

        alpha1 = -sec[4]

        alpha2 = -sec[5]

        section_rows += f"""
        <tr>
        <td style="padding:2px 10px;"><b>Stage {stage+1}</b></td>
        <td style="padding:2px 10px;">H<sub>{idx+1}</sub>(z)</td>
        <td style="padding:2px 10px;">
        b = [{sec[0]:.6f}, {sec[1]:.6f}, {sec[2]:.6f}]
        </td>
        <td style="padding:2px 10px;">
        α₁ = {alpha1:.6f}, α₂ = {alpha2:.6f}
        </td>
        </tr>
        """

    result_html.value = f"""
    <div class="cas-root">

    <div class="cas-box cas-result">

    <div class="cas-title">
    Current cascade implementation
    </div>

    <b>Section order:</b> {order_text}

    &nbsp;&nbsp;&nbsp;

    <b>Maximum |y<sub>direct</sub>[n] − y<sub>cascade</sub>[n]|:</b>
    {maximum_difference:.3e}

    <table style="margin-top:6px;font-size:12.5px;border-collapse:collapse;">
    {section_rows}
    </table>

    </div>

    </div>
    """

    # --------------------------------------------------------
    # REDRAW EXISTING CANVASES ONLY
    # --------------------------------------------------------

    fig1.canvas.draw_idle()

    fig2.canvas.draw_idle()

    fig3.canvas.draw_idle()

    fig4.canvas.draw_idle()

# ============================================================
# OBSERVER
# ============================================================

order_dropdown.observe(update,names='value')

# ============================================================
# DISPLAY
# ============================================================

display(result_html)

display(fig1.canvas)

display(controls)

display(fig2.canvas)

display(fig3.canvas)

display(fig4.canvas)

# ============================================================
# INITIAL UPDATE
# ============================================================

update()